# Claude Model Usage for Research
## A Practical Lecture for Healthcare AI Researchers

---

### Agenda
1. **Prompting Claude: Simple to Complex** — API basics, system prompts, multi-turn, thinking, tools, structured outputs
2. **VS Code Extension & Claude Code CLI** — IDE integration, slash commands, agentic coding
3. **Advanced Healthcare Research Usage** — FAERS/ADR analysis, batch processing, RAG, citations, HIPAA

---
*Prerequisites: Python 3.9+, `pip install anthropic`*

# Part 1: Prompting Claude — Simple to Complex

---

### Claude Model Family (2026)

| Model | ID | Context | Input $/1M | Output $/1M | Best For |
|---|---|---|---|---|---|
| Fable 5 | `claude-fable-5` | 1M | $10 | $50 | Frontier reasoning |
| **Opus 4.8** | `claude-opus-4-8` | 1M | $5 | $25 | **Default: complex research** |
| Sonnet 4.6 | `claude-sonnet-4-6` | 1M | $3 | $15 | Balanced speed/quality |
| Haiku 4.5 | `claude-haiku-4-5` | 200K | $1 | $5 | High-throughput classification |

> **Rule of thumb**: Start with Opus 4.8 for research. Use Haiku for batch classification tasks.

In [ ]:
# Setup — run once per session
import anthropic
import os
import json

# Set your API key
# export ANTHROPIC_API_KEY="sk-ant-..."
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment
print("Anthropic SDK version:", anthropic.__version__)

## 1.1 — The Simplest Possible Request

Three required fields:
- `model` — which Claude model
- `max_tokens` — output budget (not a target, a ceiling)
- `messages` — list of `{role, content}` dicts

```python
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    messages=[{"role": "user", "content": "What is pharmacovigilance?"}]
)
print(response.content[0].text)
```

In [ ]:
# 1.1 — Simple one-shot call
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=512,
    messages=[
        {"role": "user", "content": "What is pharmacovigilance? Give a 2-sentence definition."}
    ]
)

print(response.content[0].text)
print(f"\n--- Token usage ---")
print(f"Input:  {response.usage.input_tokens} tokens")
print(f"Output: {response.usage.output_tokens} tokens")

## 1.2 — System Prompts: Setting Persona & Context

The `system` parameter sets persistent instructions Claude follows throughout the conversation.

**Good system prompts:**
- Define the AI's role and expertise
- Specify output format preferences
- Set domain constraints
- Establish tone and vocabulary

```python
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    system="You are an expert pharmacovigilance scientist. Always cite MedDRA terminology.",
    messages=[{"role": "user", "content": "..."}]
)
```

In [ ]:
# 1.2 — System prompt example
PHARMACOVIGILANCE_SYSTEM = """You are an expert pharmacovigilance scientist with deep knowledge of:
- FDA Adverse Event Reporting System (FAERS)
- MedDRA terminology (SOC, HLGT, HLT, PT hierarchy)
- Drug safety signal detection methods
- ICH E2B guidelines

When discussing adverse events, always use MedDRA Preferred Terms (PT).
Be concise and precise. Use medical terminology."""

response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=512,
    system=PHARMACOVIGILANCE_SYSTEM,
    messages=[
        {"role": "user", "content": "What are the main cardiac adverse events associated with azithromycin?"}
    ]
)

print(response.content[0].text)

## 1.3 — Multi-Turn Conversations

Claude's API is **stateless** — you send the full conversation history every time.

```
Turn 1: [user_msg_1]                  → assistant_reply_1
Turn 2: [user_msg_1, reply_1, user_msg_2]  → assistant_reply_2
Turn 3: [user_msg_1, reply_1, ..., user_msg_3] → assistant_reply_3
```

**Key pattern:** Maintain a `history` list; append each exchange.

> **Cost insight:** Each turn re-sends the entire history. Use **prompt caching** to save up to 90% on repeated context (shown later).

In [ ]:
# 1.3 — Multi-turn conversation helper
def chat(history, user_message, system=None, model="claude-opus-4-8", max_tokens=512):
    """Send one turn; return (assistant_text, updated_history)."""
    history = history + [{"role": "user", "content": user_message}]
    kwargs = {"model": model, "max_tokens": max_tokens, "messages": history}
    if system:
        kwargs["system"] = system
    response = client.messages.create(**kwargs)
    reply = response.content[0].text
    history = history + [{"role": "assistant", "content": reply}]
    return reply, history

# Simulate a multi-turn research conversation
history = []

reply, history = chat(history, "What MedDRA SOC covers cardiovascular adverse events?",
                      system=PHARMACOVIGILANCE_SYSTEM)
print("Turn 1:", reply[:200], "...\n")

reply, history = chat(history, "List 5 important PT codes within that SOC.")
print("Turn 2:", reply[:300], "...\n")

reply, history = chat(history, "Which of those are most commonly reported in FAERS for statins?")
print("Turn 3:", reply[:300])

print(f"\nConversation turns: {len(history)//2}")

## 1.4 — Streaming for Long Outputs

Use streaming when:
- Generating long documents or reports
- Building interactive UIs
- Avoiding request timeouts (common for `max_tokens > 4000`)

```python
with client.messages.stream(...) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
```

`.get_final_message()` gives you the complete response object after the stream ends.

In [ ]:
# 1.4 — Streaming response
print("Streaming response:\n" + "-"*40)

with client.messages.stream(
    model="claude-opus-4-8",
    max_tokens=512,
    system=PHARMACOVIGILANCE_SYSTEM,
    messages=[{
        "role": "user",
        "content": "Write a brief summary of disproportionality analysis methods in pharmacovigilance (PRR, ROR, BCPNN)."
    }]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

final = stream.get_final_message()
print(f"\n\nTotal output tokens: {final.usage.output_tokens}")

## 1.5 — Adaptive Thinking (Extended Reasoning)

Gives Claude a **private reasoning scratchpad** before responding.

| Setting | When to use |
|---|---|
| `effort: "low"` | Fast classification, simple extraction |
| `effort: "medium"` | Default — general reasoning tasks |
| `effort: "high"` | Complex multi-step reasoning |
| `effort: "xhigh"` | Hardest research questions, novel analysis |
| `effort: "max"` | Maximum reasoning (highest cost) |

**Key insight**: For ADR detection with multi-hop reasoning across drug classes, use `effort: "high"` or `"xhigh"`.

In [ ]:
# 1.5 — Adaptive thinking with effort level
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=8000,
    thinking={"type": "adaptive", "display": "summarized"},
    output_config={"effort": "high"},
    system=PHARMACOVIGILANCE_SYSTEM,
    messages=[{
        "role": "user",
        "content": """
        A patient is on metformin (T2DM), atorvastatin (dyslipidemia), and lisinopril (HTN).
        They report muscle weakness and dark urine.
        
        Reason through which drug(s) are most likely responsible, considering:
        1. Known ADR profiles of each drug
        2. Drug-drug interaction potential
        3. MedDRA PT codes for the reported symptoms
        4. Recommended next steps for the clinician
        """
    }]
)

# The response may include a thinking block followed by the answer
for block in response.content:
    if block.type == "thinking":
        print("[REASONING SUMMARY]")
        print(block.thinking[:500] if block.thinking else "(omitted)")
        print()
    elif block.type == "text":
        print("[ANSWER]")
        print(block.text)

## 1.6 — Tool Use

Claude can call your functions by describing them in JSON Schema.

**Two modes:**
1. **Manual loop** — you detect `tool_use` stop reason, call the function, send back `tool_result`
2. **SDK Tool Runner** — decorator-based, handles the loop automatically

```
Request → Claude decides to call tool → Your code executes function
         ← Send tool result back ←     → Claude continues with result
```

**Server-side tools** (no code needed on your end):
- `web_search` — real-time web search
- `code_execution` — Python sandbox
- `computer_use` — desktop automation

In [ ]:
# 1.6a — Manual tool use loop (FAERS lookup example)
import random

# Simulated FAERS database lookup
def lookup_faers_reports(drug_name: str, adverse_event: str) -> dict:
    """Simulate FAERS lookup — replace with real DB query."""
    mock_counts = {"atorvastatin": {"myopathy": 4821, "rhabdomyolysis": 892, "myalgia": 12450}}
    counts = mock_counts.get(drug_name.lower(), {})
    count = counts.get(adverse_event.lower(), random.randint(10, 500))
    return {"drug": drug_name, "adverse_event": adverse_event, "report_count": count, "source": "FAERS 2024Q4"}

TOOLS = [{
    "name": "lookup_faers_reports",
    "description": "Query the FAERS database for adverse event report counts for a drug-event pair.",
    "input_schema": {
        "type": "object",
        "properties": {
            "drug_name": {"type": "string", "description": "Drug name (generic)"},
            "adverse_event": {"type": "string", "description": "MedDRA Preferred Term"}
        },
        "required": ["drug_name", "adverse_event"]
    }
}]

def run_tool_loop(user_message, tools=TOOLS, system=None):
    messages = [{"role": "user", "content": user_message}]
    kwargs = {"model": "claude-opus-4-8", "max_tokens": 1024, "tools": tools, "messages": messages}
    if system:
        kwargs["system"] = system

    while True:
        response = client.messages.create(**kwargs)
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if hasattr(b, "text"))

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  ► Calling tool: {block.name}({block.input})")
                    result = lookup_faers_reports(**block.input)
                    print(f"  ◄ Result: {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            messages.append({"role": "user", "content": tool_results})

print("Query: Compare FAERS report counts for atorvastatin myopathy vs rhabdomyolysis\n")
answer = run_tool_loop(
    "Look up FAERS report counts for atorvastatin myopathy and atorvastatin rhabdomyolysis, then compare them.",
    system=PHARMACOVIGILANCE_SYSTEM
)
print("\nFinal answer:", answer)

## 1.7 — Structured Outputs (JSON Schema)

Force Claude to return valid, schema-conforming JSON — ideal for:
- Extracting entities from clinical text
- Building downstream data pipelines
- Classification with confidence scores

Two approaches:
1. **`output_config.format`** — built-in schema enforcement
2. **`client.messages.parse()`** — validates + auto-parses response

> No more regex hacks on JSON output!

In [ ]:
# 1.7 — Structured output: extract ADR information from clinical text
ADR_EXTRACTION_SCHEMA = {
    "type": "object",
    "properties": {
        "patient_age": {"type": ["integer", "null"], "description": "Patient age in years"},
        "patient_sex": {"type": ["string", "null"], "enum": ["M", "F", "Unknown", None]},
        "suspect_drugs": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "dose": {"type": ["string", "null"]},
                    "route": {"type": ["string", "null"]}
                },
                "required": ["name"]
            }
        },
        "adverse_events": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "meddra_pt": {"type": "string", "description": "MedDRA Preferred Term"},
                    "meddra_soc": {"type": "string", "description": "System Organ Class"},
                    "severity": {"type": "string", "enum": ["mild", "moderate", "severe", "life-threatening"]},
                    "outcome": {"type": ["string", "null"]}
                },
                "required": ["meddra_pt", "severity"]
            }
        },
        "causal_assessment": {
            "type": "string",
            "enum": ["certain", "probable", "possible", "unlikely", "unclassifiable"]
        }
    },
    "required": ["suspect_drugs", "adverse_events", "causal_assessment"]
}

CLINICAL_TEXT = """
Case report: 67-year-old male with hypertension and hyperlipidemia. 
Started on atorvastatin 40mg PO daily 6 weeks ago. Presents with 
progressive proximal muscle weakness and myalgia for 2 weeks. 
CK level: 8,400 U/L. Urine dark brown. Atorvastatin held. 
Diagnosed with statin-induced rhabdomyolysis.
"""

response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    system="You are a pharmacovigilance expert. Extract structured ADR information from clinical text.",
    messages=[{"role": "user", "content": f"Extract ADR information from this case:\n{CLINICAL_TEXT}"}],
    output_config={"format": {"type": "json_schema", "json_schema": {"name": "adr_report", "schema": ADR_EXTRACTION_SCHEMA}}}
)

extracted = json.loads(response.content[0].text)
print(json.dumps(extracted, indent=2))

## 1.8 — Prompt Caching (Cost Optimization)

When the same large context is reused across many calls, cache it.

**Up to 90% cost savings** on cached tokens.

| Request type | Cost multiplier |
|---|---|
| Cache write | 1.25× normal |
| Cache read | 0.10× normal (90% off!) |

**Use case**: System prompt with 10,000-token drug knowledge base, queried 1,000 times:
- Without caching: 10M input tokens × $5 = **$50**
- With caching: 1 write + 999 reads ≈ **$5.50** (89% savings)

In [ ]:
# 1.8 — Prompt caching: cache a large drug knowledge base
DRUG_KNOWLEDGE_BASE = """
# Drug Safety Knowledge Base

## Statins (HMG-CoA Reductase Inhibitors)
Key ADRs: Myopathy, rhabdomyolysis, elevated liver enzymes, headache
Mechanism: Inhibits cholesterol synthesis, may affect CoQ10 levels in muscle
Risk factors: High dose, renal impairment, CYP3A4 inhibitors (for some statins)
Monitoring: CK levels, LFTs at baseline and periodically
Signal threshold (PRR): myopathy PRR >2 considered signal

## ACE Inhibitors
Key ADRs: Dry cough (10-15% incidence), angioedema (0.1-0.7%), hyperkalemia
Mechanism: Bradykinin accumulation → cough; may precipitate angioedema
Contraindications: Bilateral renal artery stenosis, pregnancy
MedDRA PT codes: Cough (10011224), Angioedema (10002424), Hyperkalaemia (10020647)

## Fluoroquinolones
Key ADRs: Tendinopathy, tendon rupture (Achilles), QT prolongation, peripheral neuropathy
Black box warnings: Tendon rupture, peripheral neuropathy, CNS effects
Risk factors for tendon rupture: Age >60, corticosteroid use, renal failure
MedDRA PT: Tendon rupture (10043422), QT prolongation (10037641)

## Metformin
Key ADRs: GI disturbances (nausea, diarrhea), lactic acidosis (rare, 0.03/1000 PY)
Contraindications: eGFR <30, iodinated contrast (hold 48h), hepatic impairment
MedDRA PT: Lactic acidosis (10023676), Nausea (10028813)

## Anticoagulants (Warfarin, DOACs)
Key ADRs: Bleeding (GI, intracranial), drug interactions (warfarin)
Monitoring: INR for warfarin (target 2-3 for AF), renal function for DOACs
MedDRA SOC: Blood and lymphatic system disorders; Vascular disorders
"""  # In real use, this could be 50,000+ tokens of drug data

# Use cache_control to mark the system prompt for caching
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=512,
    system=[
        {
            "type": "text",
            "text": "You are a pharmacovigilance expert. Use the drug safety database below.\n\n" + DRUG_KNOWLEDGE_BASE,
            "cache_control": {"type": "ephemeral"}  # Cache this block!
        }
    ],
    messages=[{
        "role": "user",
        "content": "What MedDRA PT code corresponds to ACE inhibitor-induced cough and what is its incidence?"
    }]
)

print(response.content[0].text)
print(f"\nCache write tokens: {response.usage.cache_creation_input_tokens}")
print(f"Cache read tokens:  {response.usage.cache_read_input_tokens}")
print(f"Regular input tokens: {response.usage.input_tokens}")

---
# Part 2: VS Code Extension & Claude Code CLI

## What is Claude Code?

An **AI coding assistant** that lives in your terminal and IDE — it reads your entire codebase and acts on it.

### Installation
```bash
# Install globally via npm
npm install -g @anthropic-ai/claude-code

# VS Code: search "Claude Code" in Extensions marketplace
# Then authenticate:
claude auth
```

### Key Principle
Claude Code is **agentic** — it can read files, run shell commands, write code, and iterate. You describe *what* you want; it figures out *how*.

## 2.1 — Core CLI Features

### Interactive Mode (default)
```bash
cd ~/Barn/GQ/ADR
claude                              # Start interactive session
```

### One-shot Commands
```bash
claude "Explain what FAERS_Prep.ipynb does"          # Non-interactive
claude -p "Review this Python file" --file utils.py  # With file context
```

### Piping
```bash
cat adr_detection.py | claude "Find potential bugs"
git diff | claude "Write a commit message"
```

## 2.2 — Slash Commands Reference

| Command | What it does |
|---|---|
| `/help` | Show all available commands |
| `/clear` | Reset conversation context |
| `/compact` | Summarize and compress conversation |
| `/cost` | Show token usage and cost |
| `/model` | Switch models mid-session |
| `/fast` | Toggle Fast mode (Opus with faster output) |
| `/plan` | Enter planning mode — Claude presents a plan first |
| `/review` | Review code changes |
| `/mcp` | Manage MCP server connections |
| `/add-dir` | Add a directory to Claude's context |

### For Jupyter Research Workflows
```
/plan  → Review the plan before Claude touches any code
/cost  → Track API spending per session
/compact → Keep long research sessions alive without losing context
```

## 2.3 — VS Code Extension Workflow

### The Difference from Copilot

| Feature | GitHub Copilot | Claude Code |
|---|---|---|
| Scope | Current file | Entire repo |
| Tool use | Code completion only | Reads, writes, runs terminal |
| Context | Line-level | Project-level |
| Interaction | Inline completions | Conversational agent |

### Common VS Code Workflows
1. **Debug a failing test**: Open test output → ask Claude what's wrong
2. **Refactor a module**: "Refactor FAERS_Prep.py to use pandas 2.0 API"
3. **Write documentation**: Select a function → "Document this with a docstring"
4. **Understand legacy code**: "Explain what this embedding pipeline does"
5. **Generate unit tests**: "Write pytest tests for the lookup_faers_reports function"

## 2.4 — MCP: Model Context Protocol

MCP lets Claude connect to **external data sources and tools** via a standard protocol.

### Configure in `.claude/settings.json`
```json
{
  "mcpServers": {
    "lancedb": {
      "command": "python",
      "args": ["-m", "mcp_lancedb_server"],
      "env": {"DB_PATH": "/home/dada/Barn/GQ/ADR/trn_lancedb"}
    },
    "faers-db": {
      "command": "python",
      "args": ["-m", "mcp_faers_server"]
    }
  }
}
```

Once connected, Claude can query your LanceDB vector store or FAERS SQL database directly during a conversation — without any copy-paste!

## 2.5 — CLAUDE.md: Project Memory

Create a `CLAUDE.md` file at the root of your project. Claude reads it at the start of every session.

```markdown
# ADR Research Project

## Project Overview
FDA Adverse Drug Reaction detection using FAERS data.
Main pipeline: FAERS_Prep.ipynb → finetune_BGE_base.ipynb → ADR_LanceDB.ipynb

## Data
- Raw FAERS quarterly files in /home/dada/Barn/GQ/ADR/faers_new/
- Processed data in adr_all_new_up2_26q1.pkl
- Vector DB in trn_lancedb/ (BGE-M3 embeddings)

## Key Conventions
- MedDRA PT codes are lowercase strings (e.g. 'myopathy')
- Drug names use generic names (FDA UNII preferred)
- All models save checkpoints to bge_adr_base*/

## Do Not
- Modify raw FAERS files (they are archives)
- Use pandas < 2.0 API
```

> This saves you from explaining your project every session!

---
# Part 3: Advanced Healthcare Research Usage

## Overview: Claude + Your ADR Research Pipeline

```
FAERS Raw Data → FAERS_Prep → BGE-M3 Embeddings → LanceDB
                                    ↓
               Claude (structured extraction, classification, analysis)
                                    ↓
              ADR Signal Detection → Publications → Clinical Decisions
```

### What Claude adds to your pipeline:
1. **Clinical text extraction** — pull structured data from free-text adverse event narratives
2. **ADR classification** — zero-shot / few-shot classification with MedDRA codes
3. **Literature analysis** — summarize papers, extract drug-event pairs
4. **Batch processing** — process thousands of reports at 50% lower cost
5. **RAG augmentation** — Claude + your vector DB for grounded answers

## 3.1 — Files API: Upload Once, Query Many Times

The Files API lets you upload large documents (PDFs, CSVs) **once** and reference them by ID in multiple requests — no re-uploading.

```python
# Upload once
file = client.beta.files.upload(
    file=("paper.pdf", open("paper.pdf", "rb"), "application/pdf")
)
file_id = file.id  # Save this!

# Use many times
client.beta.messages.create(
    model="claude-opus-4-8",
    messages=[{"role": "user", "content": [
        {"type": "document", "source": {"type": "file", "file_id": file_id},
         "citations": {"enabled": True}},  # Get quotes + page references!
        {"type": "text", "text": "Summarize the methodology"}
    ]}],
    betas=["files-api-2025-04-14"]
)
```

In [ ]:
# 3.1 — Files API: analyze a drug safety PDF (demonstration with text fallback)
import os

# For demo: create a text-based "case series" document
CASE_SERIES = b"""Drug Safety Case Series: Statin-Associated Myopathy

Background:
Statin-associated myopathy (SAM) encompasses a spectrum from mild myalgia to potentially 
fatal rhabdomyolysis. Incidence varies: myalgia 5-10%, myopathy (CK>10xULN) 0.1%, 
rhabdomyolysis 0.01-0.1 per 10,000 patient-years.

Case 1: 72yo male, atorvastatin 80mg + clarithromycin (CYP3A4 inhibitor)
Presentation: Severe proximal weakness, CK 45,000 U/L, myoglobinuria
Outcome: Hospitalization, statin + clarithromycin discontinued, renal function normalized

Case 2: 58yo female, rosuvastatin 20mg, hypothyroidism (TSH 12)
Presentation: Progressive myalgia 3 months after statin initiation, CK 3,200 U/L
Outcome: Levothyroxine dose adjustment, CK normalized without statin discontinuation

Case 3: 65yo male, simvastatin 40mg, niacin 1g/day, ESRD on dialysis
Presentation: Rhabdomyolysis CK 78,000 U/L, acute-on-chronic kidney failure
Outcome: ICU admission, hemodialysis, complete discontinuation of statin + niacin

Risk Factors Identified:
- High-dose statins (especially simvastatin 80mg)
- Drug interactions: CYP3A4 inhibitors (macrolides, azoles, amiodarone)
- Hypothyroidism (often undiagnosed)
- Renal failure (reduces statin clearance)
- Older age, female sex, Asian ethnicity
- SLCO1B1 gene variant (c.521T>C)
"""

# Upload as a plain text file
file_obj = client.beta.files.upload(
    file=("case_series.txt", CASE_SERIES, "text/plain")
)
print(f"Uploaded file ID: {file_obj.id}")

# Analyze with citations enabled
response = client.beta.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    system="You are a pharmacovigilance expert. Analyze case series and extract structured insights.",
    messages=[{"role": "user", "content": [
        {
            "type": "document",
            "source": {"type": "file", "file_id": file_obj.id},
            "citations": {"enabled": True},
            "title": "Statin Myopathy Case Series"
        },
        {"type": "text", "text": "List all drug-drug interactions mentioned and their clinical consequences."}
    ]}],
    betas=["files-api-2025-04-14"]
)

for block in response.content:
    if hasattr(block, 'text'):
        print(block.text)
    elif block.type == 'citation':
        print(f"  [Citation: {block.cited_text[:80]}...]")

## 3.2 — Batch Processing FAERS Reports

The **Message Batches API** processes thousands of reports asynchronously at **50% lower cost**.

```
Normal API: 10,000 reports × $0.005/report = $50
Batch API:  10,000 reports × $0.0025/report = $25  (same quality!)
```

**Ideal for:**
- Processing quarterly FAERS updates (100K+ reports)
- Re-coding legacy adverse event narratives
- Validating MedDRA term assignments

**Trade-off:** Batches complete within 24 hours (not real-time).

In [ ]:
# 3.2 — Batch processing FAERS adverse event narratives
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

# Simulated FAERS adverse event narratives (would come from your adr_all_new.pkl)
FAERS_NARRATIVES = [
    {"case_id": "CASE001", "narrative": "Patient developed severe chest pain and shortness of breath 2 days after starting amiodarone 200mg for atrial fibrillation."},
    {"case_id": "CASE002", "narrative": "55yo female experienced sudden onset of joint swelling and skin rash following influenza vaccination."},
    {"case_id": "CASE003", "narrative": "Patient on warfarin for DVT presented with gum bleeding and extensive bruising after adding clarithromycin for pneumonia."},
    {"case_id": "CASE004", "narrative": "Type 2 diabetic on metformin 1000mg BID developed severe abdominal cramps and watery diarrhea that resolved after dose reduction."},
    {"case_id": "CASE005", "narrative": "72yo male on lisinopril for heart failure developed sudden throat swelling requiring emergency intubation."}
]

BATCH_SYSTEM = """You are a pharmacovigilance expert. For each adverse event narrative, extract:
1. Primary MedDRA Preferred Term (PT)
2. System Organ Class (SOC)
3. Causality assessment (certain/probable/possible/unlikely)
4. Seriousness (serious/non-serious)
Return as JSON only."""

# Create batch requests
batch_requests = [
    Request(
        custom_id=case["case_id"],
        params=MessageCreateParamsNonStreaming(
            model="claude-haiku-4-5",  # Haiku for cost efficiency on batch classification
            max_tokens=256,
            system=BATCH_SYSTEM,
            messages=[{"role": "user", "content": case["narrative"]}]
        )
    )
    for case in FAERS_NARRATIVES
]

# Submit batch
batch = client.messages.batches.create(requests=batch_requests)
print(f"Batch created: {batch.id}")
print(f"Status: {batch.processing_status}")
print(f"Requests: {batch.request_counts}")
print(f"\nIn production: poll status, then retrieve results when complete.")
print(f"Results available via: client.messages.batches.results(batch.id)")

In [ ]:
# 3.2b — Poll and retrieve batch results (run after batch completes)
import time

def wait_and_get_batch_results(batch_id, poll_interval=5, max_wait=120):
    """Poll until batch completes and return results dict keyed by custom_id."""
    elapsed = 0
    while elapsed < max_wait:
        status = client.messages.batches.retrieve(batch_id)
        print(f"Status: {status.processing_status} | " 
              f"Succeeded: {status.request_counts.succeeded} | "
              f"Processing: {status.request_counts.processing}")
        
        if status.processing_status == "ended":
            results = {}
            for result in client.messages.batches.results(batch_id):
                if result.result.type == "succeeded":
                    text = result.result.message.content[0].text
                    try:
                        results[result.custom_id] = json.loads(text)
                    except json.JSONDecodeError:
                        results[result.custom_id] = {"raw": text}
                else:
                    results[result.custom_id] = {"error": result.result.error.message}
            return results
        
        time.sleep(poll_interval)
        elapsed += poll_interval
    
    return None

# Uncomment to actually wait for results:
# results = wait_and_get_batch_results(batch.id)
# if results:
#     for case_id, result in results.items():
#         print(f"\n{case_id}:")
#         print(json.dumps(result, indent=2))

print("Batch polling helper defined. Batches may take up to 24h in production.")

## 3.3 — RAG: Claude + Your Vector Database

Combine your BGE-M3 + LanceDB pipeline with Claude for **grounded, citable answers**.

```
User Query
    ↓
BGE-M3 Embedding → LanceDB Vector Search → Top-K Relevant Chunks
    ↓
Claude (query + retrieved chunks) → Grounded Answer with Citations
```

**Why this beats pure LLM:**
- Answers grounded in your proprietary FAERS data
- Citations point to specific case reports or literature
- No hallucinated statistics
- Can answer questions about drugs not in Claude's training data

In [ ]:
# 3.3 — RAG pattern: Claude + LanceDB for ADR Q&A
# This demonstrates the pattern — connect to your actual LanceDB instance

def claude_rag_query(query: str, retrieved_docs: list[dict], model="claude-opus-4-8") -> str:
    """Run a RAG query: inject retrieved FAERS docs into Claude's context."""
    # Format retrieved documents as numbered context
    context_blocks = []
    for i, doc in enumerate(retrieved_docs, 1):
        context_blocks.append(
            f"[Document {i}]\n"
            f"Drug: {doc.get('drug', 'Unknown')}\n"
            f"Event: {doc.get('adverse_event', 'Unknown')}\n"
            f"Report count: {doc.get('report_count', 'N/A')}\n"
            f"Narrative: {doc.get('narrative', '')}\n"
        )
    context = "\n---\n".join(context_blocks)

    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system="""You are a pharmacovigilance expert. Answer questions using ONLY the provided FAERS documents.
Always cite specific documents (e.g., [Document 2]) when making claims.
If the documents don't contain enough information, say so explicitly.""",
        messages=[{
            "role": "user",
            "content": f"FAERS Database Context:\n{context}\n\nQuestion: {query}"
        }]
    )
    return response.content[0].text

# Simulated retrieval results (in practice: lancedb_table.search(embed(query)).limit(5).to_list())
mock_retrieved = [
    {"drug": "atorvastatin", "adverse_event": "rhabdomyolysis", "report_count": 892,
     "narrative": "High-dose atorvastatin combined with clarithromycin led to severe rhabdomyolysis in multiple cases."},
    {"drug": "atorvastatin", "adverse_event": "myopathy", "report_count": 4821,
     "narrative": "Statin-induced myopathy is the most commonly reported musculoskeletal ADR with atorvastatin."},
    {"drug": "simvastatin", "adverse_event": "rhabdomyolysis", "report_count": 1205,
     "narrative": "Simvastatin 80mg has been associated with rhabdomyolysis; dose capped at 40mg in most guidelines."},
]

answer = claude_rag_query(
    query="Which statins are most associated with rhabdomyolysis and what are the risk factors?",
    retrieved_docs=mock_retrieved
)
print(answer)

## 3.4 — ADR Classification: Few-Shot with Claude

Use Claude to assign MedDRA PT codes to free-text adverse event descriptions.

**Advantages over rule-based:**
- Handles misspellings and abbreviations
- Maps informal patient language to clinical terms
- Multi-label: one narrative can map to multiple PTs

**Strategy:**
1. **Few-shot examples** — 3-5 labeled examples in the system prompt
2. **Confidence scores** — ask Claude to rate certainty
3. **Human review queue** — flag low-confidence assignments

In [ ]:
# 3.4 — ADR classification with few-shot examples
FEW_SHOT_SYSTEM = """You are a MedDRA coding expert. Assign MedDRA Preferred Terms (PTs) to adverse event descriptions.

## Examples
Input: "patient feels dizzy when standing up"
Output: {"pts": [{"pt": "Orthostatic hypotension", "code": "10031127", "confidence": 0.95},
                  {"pt": "Dizziness", "code": "10013573", "confidence": 0.85}]}

Input: "heart beating fast and irregular"
Output: {"pts": [{"pt": "Palpitations", "code": "10033557", "confidence": 0.90},
                  {"pt": "Atrial fibrillation", "code": "10003658", "confidence": 0.70}]}

Input: "severe stomach pain, couldn't eat for 2 days"
Output: {"pts": [{"pt": "Abdominal pain", "code": "10000081", "confidence": 0.95},
                  {"pt": "Decreased appetite", "code": "10011912", "confidence": 0.80}]}

Return ONLY valid JSON matching the format above."""

# Batch classify multiple reports
test_narratives = [
    "patient couldn't sleep and was anxious all night",
    "developed yellow skin and eyes, liver enzymes very high",
    "severe muscle pain and cola-colored urine",
    "swollen legs and difficulty breathing when lying flat"
]

results = []
for narrative in test_narratives:
    response = client.messages.create(
        model="claude-haiku-4-5",  # Fast + cheap for bulk classification
        max_tokens=256,
        system=FEW_SHOT_SYSTEM,
        messages=[{"role": "user", "content": narrative}]
    )
    try:
        result = json.loads(response.content[0].text)
        results.append({"narrative": narrative[:60], "codes": result["pts"]})
    except json.JSONDecodeError:
        results.append({"narrative": narrative[:60], "raw": response.content[0].text})

for r in results:
    print(f"\nNarrative: '{r['narrative']}...'")
    for pt in r.get('codes', []):
        print(f"  → {pt['pt']} ({pt.get('code', 'N/A')}) [{pt['confidence']:.0%} confidence]")

## 3.5 — Literature Analysis with Citations

When analyzing medical literature, citations are critical for:
- Verifying Claude's claims against source text
- Audit trails in clinical research
- Avoiding hallucinated statistics

```python
# Enable citations on documents
{
    "type": "document",
    "source": {"type": "file", "file_id": uploaded.id},
    "citations": {"enabled": True}  # ← This is the key flag
}
```

Claude will respond with inline citations like:
```
"The incidence of myopathy was 0.1% [Source 1, p.3]..."
```

In [ ]:
# 3.5 — Literature analysis: extract drug-event pairs from abstract
MEDICAL_ABSTRACT = """
Title: Fluoroquinolone-Associated Tendinopathy: A Systematic Review

Background: Fluoroquinolone antibiotics are associated with tendinopathy and tendon rupture.

Methods: We systematically reviewed case reports and cohort studies from 1983-2024 
involving fluoroquinolone-associated musculoskeletal adverse events (n=847 cases).

Results: Achilles tendon was affected in 68% of cases. Ciprofloxacin was the most 
commonly implicated agent (47%), followed by levofloxacin (31%) and moxifloxacin (12%).
Median onset was 14 days (range 1-6 months). Risk factors: age >60 years (OR 3.4, 
95% CI 2.1-5.5), concurrent corticosteroid use (OR 7.2, 95% CI 4.8-10.8), and renal 
failure (OR 2.9, 95% CI 1.7-4.9). Tendon rupture occurred in 22% of cases.

Conclusions: All fluoroquinolones carry tendinopathy risk. Risk is highest with 
ciprofloxacin in elderly patients on corticosteroids. Black box warning is warranted.
"""

# Upload document
abstract_file = client.beta.files.upload(
    file=("abstract.txt", MEDICAL_ABSTRACT.encode(), "text/plain")
)

response = client.beta.messages.create(
    model="claude-opus-4-8",
    max_tokens=1024,
    system="Extract structured drug safety information from medical literature. Be precise and cite specific statistics.",
    messages=[{"role": "user", "content": [
        {
            "type": "document",
            "source": {"type": "file", "file_id": abstract_file.id},
            "citations": {"enabled": True},
            "title": "Fluoroquinolone Tendinopathy Review"
        },
        {"type": "text", "text": """
        Extract: 1) Which drugs are implicated and their frequencies
        2) Risk factors with odds ratios
        3) MedDRA PT codes for the reported events
        Cite specific statistics from the document.
        """}
    ]}],
    betas=["files-api-2025-04-14"]
)

print(response.content[0].text)

## 3.6 — HIPAA & Data Privacy Considerations

### Before Using Claude with Patient Data

| Scenario | Status | What to do |
|---|---|---|
| De-identified research data (FAERS) | ✅ Generally OK | Standard API use |
| IRB-approved de-identified data | ✅ OK with BAA | Sign BAA with Anthropic |
| PHI (identified patient records) | ⚠️ Requires BAA | Enterprise API only |
| Clinical trial data with patient IDs | ❌ Without BAA | De-identify first |

### Safe Practices
1. **De-identify before sending**: Remove name, DOB, MRN, zip codes
2. **Use FAERS (already de-identified)**: Public dataset is safe
3. **Literature analysis**: Always safe — no patient data
4. **Enterprise plan + BAA**: Required for any PHI
5. **Zero data retention**: Available for sensitive workloads

> FAERS data you use for research is already de-identified under 21 CFR Part 314 — you can use it with the standard API.

## 3.7 — Agentic Research Workflow

Combine everything into an end-to-end research agent:

```
Research Question
    ↓
Agent Step 1: Web search for recent publications
    ↓
Agent Step 2: Upload papers to Files API
    ↓
Agent Step 3: Extract drug-event pairs with citations
    ↓  
Agent Step 4: Cross-reference with FAERS (via tool call to your DB)
    ↓
Agent Step 5: Generate synthesis report with confidence levels
```

**Server-side tools** (no hosting required):
- `web_search` — find recent papers automatically
- `code_execution` — compute statistics, generate plots

In [ ]:
# 3.7 — Agentic research workflow with web search + tool use
RESEARCH_TOOLS = [
    # Server-side web search (Anthropic runs this)
    {"type": "web_search_20260209"},
    # Your FAERS database tool
    {
        "name": "query_faers_db",
        "description": "Query FAERS database for drug-event report counts and PRR statistics.",
        "input_schema": {
            "type": "object",
            "properties": {
                "drug": {"type": "string"},
                "event_meddra_pt": {"type": "string"},
                "year_range": {"type": "string", "description": "e.g. '2020-2024'"}
            },
            "required": ["drug", "event_meddra_pt"]
        }
    }
]

def query_faers_db(drug: str, event_meddra_pt: str, year_range: str = "all") -> dict:
    """Stub — replace with actual LanceDB/SQL query."""
    return {
        "drug": drug, "event": event_meddra_pt,
        "n_reports": 1247, "prr": 3.8, "prr_ci_lower": 2.9, "prr_ci_upper": 5.0,
        "year_range": year_range, "signal_status": "confirmed_signal"
    }

def run_research_agent(research_question: str):
    """Run a multi-step research agent on a drug safety question."""
    messages = [{"role": "user", "content": research_question}]
    
    SYSTEM = """You are a pharmacovigilance research agent. For drug safety questions:
1. Search for recent evidence using web_search
2. Cross-reference findings with the FAERS database using query_faers_db
3. Synthesize evidence into a structured safety summary
Always quantify claims with statistics from your tool results."""

    step = 0
    while step < 6:  # Safety limit
        response = client.messages.create(
            model="claude-opus-4-8",
            max_tokens=2048,
            tools=RESEARCH_TOOLS,
            system=SYSTEM,
            messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if hasattr(b, "text"))

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"[Step {step+1}] Agent calling: {block.name}")
                    if block.name == "query_faers_db":
                        result = query_faers_db(**block.input)
                    else:
                        result = {"status": "web_search handled by Anthropic servers"}
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                          "content": json.dumps(result)})
            messages.append({"role": "user", "content": tool_results})
        step += 1
    return "Agent reached step limit"

print("Running research agent...\n")
report = run_research_agent(
    "What is the current evidence for fluoroquinolone-associated tendinopathy? "
    "Search for recent studies and check our FAERS database for 'Tendon rupture' signal strength."
)
print("\n=== RESEARCH SUMMARY ===\n")
print(report[:1500])

## 3.8 — Choosing the Right Claude Model for Healthcare Tasks

| Task | Recommended Model | Why |
|---|---|---|
| Complex ADR reasoning, drug interactions | Opus 4.8 + `effort: "high"` | Deep clinical reasoning |
| Literature summarization | Opus 4.8 | Long documents, nuanced synthesis |
| MedDRA coding (bulk) | Haiku 4.5 | Fast, cheap, sufficient for structured tasks |
| Patient narrative extraction | Sonnet 4.6 | Balanced — good NLP, moderate cost |
| FAERS batch processing (100K+) | Haiku 4.5 (Batch API) | Maximum throughput at minimum cost |
| Research agent (multi-step) | Opus 4.8 | Complex planning and tool use |

### Cost Optimization Checklist
- ☑ Use Haiku for bulk classification (10× cheaper than Opus)
- ☑ Enable prompt caching for repeated drug knowledge bases (90% savings)
- ☑ Use Batch API for non-real-time processing (50% savings)
- ☑ Upload documents once via Files API (avoid re-uploading)
- ☑ Set `max_tokens` conservatively — you're charged for actual output

# Summary & Key Takeaways

## Prompting Ladder
```
Simple call → System prompt → Multi-turn → Thinking → Tools → Agents
```

## VS Code Integration
- `CLAUDE.md` = project memory (describe your data, conventions)
- `/plan` before touching code, `/cost` to monitor spend
- MCP connects Claude to your LanceDB / SQL databases

## Healthcare Research Stack
| Layer | Tool | Purpose |
|---|---|---|
| Retrieval | LanceDB + BGE-M3 | Find relevant FAERS reports |
| Reasoning | Claude Opus 4.8 | Extract, classify, synthesize |
| Scale | Batch API + Haiku | Process quarterly FAERS updates |
| Audit | Citations + structured output | Reproducible research |

---
**Next steps:**
1. Install Claude Code: `npm install -g @anthropic-ai/claude-code`
2. Create `CLAUDE.md` in your ADR project directory
3. Try the FAERS batch classification pipeline on your next data update

## Resources & References

| Resource | URL |
|---|---|
| Anthropic API Docs | https://docs.anthropic.com |
| Python SDK | https://github.com/anthropic-ai/anthropic-sdk-python |
| Claude Code CLI | https://claude.ai/code |
| Model Pricing | https://anthropic.com/api/pricing |
| MedDRA Browser | https://www.meddra.org |
| FAERS Data | https://fis.fda.gov/extensions/FPD-QDE-FAERS |

### Install for this lecture
```bash
pip install anthropic>=0.40.0
npm install -g @anthropic-ai/claude-code
export ANTHROPIC_API_KEY="sk-ant-..."
```